In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime



parsing

In [ ]:
LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ \S+ \[(?P<time>.*?)\] '
    r'"(?P<method>\S+) (?P<endpoint>\S+) \S+" '
    r'(?P<status>\d+) (?P<size>\S+) '
    r'"(?P<referer>.*?)" "(?P<useragent>.*?)"'
)


def parse_log(filepath, label):
    rows = []
    with open(filepath) as f:
        for line in f:
            m = LOG_PATTERN.match(line)
            if not m:
                continue
            d = m.groupdict()
            
            ts = datetime.strptime(d['time'], '%d/%b/%Y:%H:%M:%S %z')
            rows.append({
                'timestamp': ts,
                'status': int(d['status']),
                'size': int(d['size']) if d['size'].isdigit() else 0,
                'useragent': d['useragent'],
                'label': label
            })
    df = pd.DataFrame(rows)
    return df


normal_df = parse_log('/normal_traffic.log', label=0)
attack_df = parse_log('/attack_traffic.log', label=1)

print(f"Normal requests: {len(normal_df)}")
print(f"Attack requests: {len(attack_df)}")


feature engineering

In [ ]:
def make_windows(df, window_seconds=10):

    df = df.sort_values('timestamp').reset_index(drop=True)

    # time between consecutive reqs
    df['gap'] = df['timestamp'].diff().dt.total_seconds().fillna(0)

    # assign each req to window
    start = df['timestamp'].min()
    df['window'] = ((df['timestamp'] - start).dt.total_seconds() // window_seconds).astype(int)



    features = []
    for win, group in df.groupby('window'):
        gaps = group['gap'].values
        features.append({
            'request_count': len(group),           
            'avg_gap': np.mean(gaps),              
            'std_gap': np.std(gaps),               
            'min_gap': np.min(gaps),
            'max_gap': np.max(gaps),
            'avg_size': group['size'].mean(),      
            'label': group['label'].iloc[0]
        })
    return pd.DataFrame(features)




normal_windows = make_windows(normal_df)
attack_windows = make_windows(attack_df)

data = pd.concat([normal_windows, attack_windows], ignore_index=True)
print(data.head())
print(f"\ntotal windows: {len(data)}")
print(f"normal windows: {len(normal_windows)}, attack windows: {len(attack_windows)}")